In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

# Check if we are currently inside the 'notebooks' folder
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [3]:
from src.config import SimConfig
from src.utils.data_processing import load_and_cache_entire_fleet
from src.utils.evaluation import VoyageBenchmarker, print_markdown_table

# Legacy Imports
from src.plants import FuelCellOnlyPlant, HybridPlant

from src.solvers import HybridSDPSolver, BaselineSDPSolver

from src.controllers import (
    build_approach,
    HybridFCLockedControl,
    HybridPolicyControl,
    HybridValueControl,
    BaselineConstantControl,
    BaselineThresholdControl,
    BaselineSDPControl
)

from src.utils.plotting import plot_benchmarker_results

config = SimConfig()
fleet_data = load_and_cache_entire_fleet(config)

# Initialize benchmarker
exclude_days = [] 
benchmarker = VoyageBenchmarker(fleet_data, config, exclude_days)

Beginning memory staging of all 14 fleet files into RAM...
 -> Day 01 successfully cached in RAM.
 -> Day 02 successfully cached in RAM.
 -> Day 03 successfully cached in RAM.
 -> Day 04 successfully cached in RAM.
 -> Day 05 successfully cached in RAM.
 -> Day 06 successfully cached in RAM.
 -> Day 07 successfully cached in RAM.
 -> Day 08 successfully cached in RAM.
 -> Day 09 successfully cached in RAM.
 -> Day 10 successfully cached in RAM.
 -> Day 11 successfully cached in RAM.
 -> Day 12 successfully cached in RAM.
 -> Day 13 successfully cached in RAM.
 -> Day 14 successfully cached in RAM.

All 14 operational days securely held in RAM. Disk I/O locked.


In [6]:
fc_only_approaches = {
    "NaiveConstantControl": build_approach(
        controller_cls=BaselineConstantControl,
        is_macro=True
    ),
    "HybridConstantControl": build_approach(
        controller_cls=BaselineConstantControl
    ),
    "NaiveThresholdControl": build_approach(
        controller_cls=BaselineThresholdControl,
        is_macro=True
    ),
    "HybridThresholfControl": build_approach(
        controller_cls=BaselineThresholdControl
    ),
    "NaiveSDPControl": build_approach(
        controller_cls=BaselineSDPControl,
        solver_cls=BaselineSDPSolver,
        is_macro=True
    ),
    "HybridSDPControl": build_approach(
        controller_cls=BaselineSDPControl,
        solver_cls=BaselineSDPSolver
    ),
}

hybrid_approaches = {
    "NaiveFCLocked": build_approach(
        controller_cls=HybridFCLockedControl,
        solver_cls=HybridSDPSolver,
        is_macro=True
    ),
    "FCLocked": build_approach(
        controller_cls=HybridFCLockedControl,
        solver_cls=HybridSDPSolver,
    ),
        "NaivePolicy": build_approach(
        controller_cls=HybridPolicyControl,
        solver_cls=HybridSDPSolver,
        is_macro=True
    ),
        "Policy": build_approach(
        controller_cls=HybridPolicyControl,
        solver_cls=HybridSDPSolver,
    ),
        "NaiveValue": build_approach(
        controller_cls=HybridValueControl,
        solver_cls=HybridSDPSolver,
        is_macro=True
    ),
        "Value": build_approach(
        controller_cls=HybridValueControl,
        solver_cls=HybridSDPSolver,
    ),
}

In [ ]:
print("--- APPROACH A: HAND-PICKED EVALUATION ---")
# Manually choose training block and test validation target
train_days = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
test_day = 14
approaches = hybrid_approaches
df_manual = benchmarker.compare_approaches(approaches, train_days, test_day)
print_markdown_table(df_manual)

--- APPROACH A: HAND-PICKED EVALUATION ---


In [ ]:
print("\n--- APPROACH B: LEAVE-ONE-OUT (Discrete Tracking vs Baseline) ---")

# Note: Running LOO on all 5 approaches might take a couple of minutes due to 4D solves.
# Let's compare the main engineering deployable hybrid against the legacy baseline.

df_loo_legacy = benchmarker.run_leave_one_out(approaches["1. FC-Only Baseline"])
display(df_loo_legacy)

df_loo_hybrid = benchmarker.run_leave_one_out(approaches["3. Hybrid (Discrete Tracking)"])
display(df_loo_hybrid)

# Visualize the day-to-day volatility
plot_benchmarker_results(df_loo_legacy, title="Leave-One-Out Cross Validation (FC-Only Baseline)", plot_type='bar')
plot_benchmarker_results(df_loo_hybrid, title="Leave-One-Out Cross Validation (Discrete Hybrid)", plot_type='bar')

In [ ]:
print("\n--- APPROACH C: FORWARD CHAINING (Learning Curve) ---")
# Evaluate how the policy improves as the agent gathers chronological data
# Using the Lookahead Optimum to see the absolute theoretical ceiling of the ship's capabilities

df_for_opt = benchmarker.run_forward_chaining(approaches["5. Hybrid (Lookahead Optimum)"])
display(df_for_opt)

plot_benchmarker_results(df_for_opt, title="Forward Chaining Learning Curve (Theoretical Optimum)", plot_type='line')